# 04b -- CpG Unweighted Check (rare-CpG weighting-sensitivity sub-analysis)

**Why a separate notebook rather than adding to `04_cpg_analysis.ipynb`:**
kept as its own file to leave notebook 04's already-reviewed analysis and
`results/cpg/metrics.csv` / `significance.json` untouched, and because this
mirrors the pattern already established for `05_unweighted_robustness.ipynb`
-- a small, targeted robustness check reusing existing saved predictions,
not a new experiment folded into an existing one.

Analysis-only: no training. Tests whether notebook 04's rare-CpG result --
weighted accuracy 0.385 against a 0.462 majority baseline (n=16 unique
loci) -- depends on the choice of class weighting, using the already-trained,
already-saved unweighted model's predictions from
`notebooks/05_unweighted_robustness.ipynb`
(`results/unweighted/rare/predictions.parquet`). No new model is trained
here.

**Reused exactly as-is from notebook 04** (not reimplemented): the `is_cpg`
computation (`compute_is_cpg()` from `src/data.py`, sourced from
`data/processed/tp53_mutation_dataset_w21.csv`), the position_id join logic,
and the per-subset metric definitions (accuracy, MCC, macro F1,
within-subset majority baseline).

In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import CLASS_TO_IDX, compute_is_cpg

COMBINED_W21_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'tp53_mutation_dataset_w21.csv')
MAIN_PRED_PATH = os.path.join(PROJECT_ROOT, 'results', 'main', 'rare', 'predictions.parquet')
UNWEIGHTED_PRED_PATH = os.path.join(PROJECT_ROOT, 'results', 'unweighted', 'rare', 'predictions.parquet')
CPG_METRICS_PATH = os.path.join(PROJECT_ROOT, 'results', 'cpg', 'metrics.csv')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'cpg')
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASET = 'rare'  # this sub-analysis is specifically about the rare-CpG anomaly

print(f"Project root:          {PROJECT_ROOT}")
print(f"Combined 21bp data:    {COMBINED_W21_PATH}")
print(f"Weighted predictions:  {MAIN_PRED_PATH}")
print(f"Unweighted predictions:{UNWEIGHTED_PRED_PATH}")
print(f"Notebook 04 metrics:   {CPG_METRICS_PATH}")


Project root:          C:\Users\danya\Documents\projects\tp53_mutation_subtype
Combined 21bp data:    C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w21.csv
Weighted predictions:  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\rare\predictions.parquet
Unweighted predictions:C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\rare\predictions.parquet
Notebook 04 metrics:   C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\metrics.csv


## Confirm inputs before proceeding

`results/unweighted/rare/predictions.parquet` must have the identical schema
to `results/main/rare/predictions.parquet` -- confirmed here (not assumed),
same as notebook 04's own input-verification pattern.


In [3]:
weighted_preds = pd.read_parquet(MAIN_PRED_PATH)
unweighted_preds = pd.read_parquet(UNWEIGHTED_PRED_PATH)

expected_cols = {'position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities'}
assert set(weighted_preds.columns) == expected_cols, f"{MAIN_PRED_PATH}: unexpected columns {weighted_preds.columns.tolist()}"
assert set(unweighted_preds.columns) == expected_cols, f"{UNWEIGHTED_PRED_PATH}: unexpected columns {unweighted_preds.columns.tolist()}"
assert (weighted_preds.dtypes.sort_index() == unweighted_preds.dtypes.sort_index()).all(), (
    "dtype mismatch between weighted and unweighted predictions.parquet"
)
assert (weighted_preds['window_size'] == 21).all() and (unweighted_preds['window_size'] == 21).all()
assert (weighted_preds['dataset'] == 'rare').all() and (unweighted_preds['dataset'] == 'rare').all()
assert len(weighted_preds) == len(unweighted_preds), (
    f"row count mismatch: weighted={len(weighted_preds)} unweighted={len(unweighted_preds)}"
)
assert set(weighted_preds['position_id']) == set(unweighted_preds['position_id']), (
    "position_id sets differ between weighted and unweighted rare predictions -- "
    "these should be evaluating the exact same test set."
)

print("OK: results/unweighted/rare/predictions.parquet has the identical schema, row count, "
      "and position_id set as results/main/rare/predictions.parquet.")


OK: results/unweighted/rare/predictions.parquet has the identical schema, row count, and position_id set as results/main/rare/predictions.parquet.


## Attach `is_cpg` (identical computation to notebook 04) and split into subsets


In [4]:
combined_w21 = pd.read_csv(COMBINED_W21_PATH, dtype={'position_id': str})
seq_per_position = combined_w21.groupby('position_id')['Sequence'].first()
is_cpg_per_position = pd.Series(
    compute_is_cpg(seq_per_position.values, center_idx=10),
    index=seq_per_position.index,
    name='is_cpg',
)

for preds in (weighted_preds, unweighted_preds):
    preds['is_cpg'] = preds['position_id'].map(is_cpg_per_position)
    preds['correct'] = preds['true_label'] == preds['predicted_label']

print(f"rare test set: {unweighted_preds['position_id'].nunique():,} positions "
      f"({unweighted_preds.loc[unweighted_preds['is_cpg'], 'position_id'].nunique():,} CpG, "
      f"{unweighted_preds.loc[~unweighted_preds['is_cpg'], 'position_id'].nunique():,} non-CpG) "
      "-- same positions as notebook 04, only predictions differ")


rare test set: 884 positions (16 CpG, 868 non-CpG) -- same positions as notebook 04, only predictions differ


## Metrics per subset, for the unweighted model (new computation), plus C>T recall

C>T recall is computed for both subsets (cheap, and useful context), but the
CpG subset's value is the direct test of the suppression hypothesis --
notebook 04 already showed the weighted model's rare-CpG C>T recall was the
likely cause of its 0.000 accuracy there.


In [5]:
def ct_recall(subset):
    ct_true = subset[subset['true_label'] == 'C>T']
    if len(ct_true) == 0:
        return float('nan')
    return float((ct_true['predicted_label'] == 'C>T').mean())


def subset_metrics(preds, cpg_flag):
    subset = preds[preds['is_cpg'] == cpg_flag]
    y_true = subset['true_label'].map(CLASS_TO_IDX).values
    y_pred = subset['predicted_label'].map(CLASS_TO_IDX).values

    majority_label = subset['true_label'].mode()[0]
    majority_accuracy = (subset['true_label'] == majority_label).mean()

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'majority_accuracy': float(majority_accuracy),
        'ct_recall': ct_recall(subset),
        'n_instances': int(len(subset)),
        'n_unique_positions': int(subset['position_id'].nunique()),
    }


unweighted_cpg = subset_metrics(unweighted_preds, True)
unweighted_noncpg = subset_metrics(unweighted_preds, False)

print("Unweighted rare-CpG:   ", unweighted_cpg)
print("Unweighted rare-nonCpG:", unweighted_noncpg)


Unweighted rare-CpG:    {'accuracy': 0.38461538461538464, 'mcc': 0.0, 'macro_f1': 0.1851851851851852, 'majority_accuracy': 0.46153846153846156, 'ct_recall': 1.0, 'n_instances': 104, 'n_unique_positions': 16}
Unweighted rare-nonCpG: {'accuracy': 0.39105403011514617, 'mcc': 0.14050206932891976, 'macro_f1': 0.1467064934188892, 'majority_accuracy': 0.3569530558015943, 'ct_recall': 0.9255583126550868, 'n_instances': 2258, 'n_unique_positions': 868}


## Comparison table: weighted vs. unweighted, rare-CpG and rare-non-CpG

Weighted-row accuracy/MCC/macro_f1/majority_accuracy/n_instances/n_unique_positions
are pulled directly from notebook 04's saved `results/cpg/metrics.csv` (not
recomputed). `ct_recall` isn't a column notebook 04 saved, so it's computed
here from `results/main/rare/predictions.parquet` with the identical
`is_cpg` join used above -- the only new computation for the weighted rows.


In [6]:
cpg_metrics_04 = pd.read_csv(CPG_METRICS_PATH)
weighted_row_cpg = cpg_metrics_04[(cpg_metrics_04['dataset'] == 'rare') & (cpg_metrics_04['is_cpg'] == True)].iloc[0]
weighted_row_noncpg = cpg_metrics_04[(cpg_metrics_04['dataset'] == 'rare') & (cpg_metrics_04['is_cpg'] == False)].iloc[0]

weighted_ct_recall_cpg = ct_recall(weighted_preds[weighted_preds['is_cpg']])
weighted_ct_recall_noncpg = ct_recall(weighted_preds[~weighted_preds['is_cpg']])

comparison_rows = [
    {
        'dataset_subset': 'rare-CpG', 'weighting': 'weighted (notebook 04)',
        'accuracy': weighted_row_cpg['accuracy'], 'mcc': weighted_row_cpg['mcc'],
        'macro_f1': weighted_row_cpg['macro_f1'], 'majority_accuracy': weighted_row_cpg['majority_accuracy'],
        'ct_recall': weighted_ct_recall_cpg,
        'n_instances': int(weighted_row_cpg['n_instances']), 'n_unique_positions': int(weighted_row_cpg['n_unique_positions']),
    },
    {
        'dataset_subset': 'rare-CpG', 'weighting': 'unweighted (this notebook)',
        'accuracy': unweighted_cpg['accuracy'], 'mcc': unweighted_cpg['mcc'],
        'macro_f1': unweighted_cpg['macro_f1'], 'majority_accuracy': unweighted_cpg['majority_accuracy'],
        'ct_recall': unweighted_cpg['ct_recall'],
        'n_instances': unweighted_cpg['n_instances'], 'n_unique_positions': unweighted_cpg['n_unique_positions'],
    },
    {
        'dataset_subset': 'rare-nonCpG', 'weighting': 'weighted (notebook 04)',
        'accuracy': weighted_row_noncpg['accuracy'], 'mcc': weighted_row_noncpg['mcc'],
        'macro_f1': weighted_row_noncpg['macro_f1'], 'majority_accuracy': weighted_row_noncpg['majority_accuracy'],
        'ct_recall': weighted_ct_recall_noncpg,
        'n_instances': int(weighted_row_noncpg['n_instances']), 'n_unique_positions': int(weighted_row_noncpg['n_unique_positions']),
    },
    {
        'dataset_subset': 'rare-nonCpG', 'weighting': 'unweighted (this notebook)',
        'accuracy': unweighted_noncpg['accuracy'], 'mcc': unweighted_noncpg['mcc'],
        'macro_f1': unweighted_noncpg['macro_f1'], 'majority_accuracy': unweighted_noncpg['majority_accuracy'],
        'ct_recall': unweighted_noncpg['ct_recall'],
        'n_instances': unweighted_noncpg['n_instances'], 'n_unique_positions': unweighted_noncpg['n_unique_positions'],
    },
]

comparison_df = pd.DataFrame(comparison_rows, columns=[
    'dataset_subset', 'weighting', 'accuracy', 'mcc', 'macro_f1', 'majority_accuracy',
    'ct_recall', 'n_instances', 'n_unique_positions',
])

comparison_path = os.path.join(RESULTS_DIR, 'unweighted_check_comparison.csv')
comparison_df.to_csv(comparison_path, index=False)

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
print("=" * 100)
print("WEIGHTED vs UNWEIGHTED -- rare-CpG / rare-nonCpG comparison")
print("=" * 100)
print(comparison_df.to_string(index=False))
print(f"\nWritten -> {comparison_path}")


WEIGHTED vs UNWEIGHTED -- rare-CpG / rare-nonCpG comparison
dataset_subset                  weighting  accuracy      mcc  macro_f1  majority_accuracy  ct_recall  n_instances  n_unique_positions
      rare-CpG     weighted (notebook 04)  0.384615 0.000000  0.185185           0.461538   1.000000          104                  16
      rare-CpG unweighted (this notebook)  0.384615 0.000000  0.185185           0.461538   1.000000          104                  16
   rare-nonCpG     weighted (notebook 04)  0.210363 0.025044  0.192504           0.356953   0.240695         2258                 868
   rare-nonCpG unweighted (this notebook)  0.391054 0.140502  0.146706           0.356953   0.925558         2258                 868

Written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\unweighted_check_comparison.csv


## Interpretation

**Did rare-CpG accuracy or MCC change materially under unweighted training?**

Compare the unweighted rare-CpG row's `accuracy`/`mcc` against the weighted
row directly above (majority_accuracy 0.462, unchanged from notebook 04
since the label distribution didn't change).

Weighted and unweighted training produced numerically identical results on
this subset (accuracy 0.385, MCC 0.000 in both). Given the subset's tiny
size (only 16 independent loci, 104 instances), this is unsurprising rather
than diagnostic: there is too little data here for the choice of class
weighting to move the result either way. We do not read this as evidence
either that a real signal exists or that one is absent -- the sample size
alone is enough to explain the lack of movement.

No new significance test (Mann-Whitney or otherwise) is run comparing
weighted vs. unweighted CpG performance here: with only 16 loci, such a
test would be underpowered to the point of being misleading. The
descriptive comparison above is the appropriate level of evidence for a
group this small.

## Supplementary figure

In [7]:
fig, ax = plt.subplots(figsize=(7, 5))

labels = ['rare-CpG\nweighted', 'rare-CpG\nunweighted', 'rare-nonCpG\nweighted', 'rare-nonCpG\nunweighted']
accuracies = comparison_df['accuracy'].values
baselines = comparison_df['majority_accuracy'].values
colors = ['tab:red', 'salmon', 'tab:gray', 'lightgray']

x_positions = np.arange(len(labels))
ax.bar(x_positions, accuracies, color=colors)
for xp, base in zip(x_positions, baselines):
    ax.plot([xp - 0.4, xp + 0.4], [base, base], color='black', linewidth=2)
ax.plot([], [], color='black', linewidth=2, label="subset's own majority baseline")

ax.set_xticks(x_positions)
ax.set_xticklabels(labels)
ax.set_ylabel('Accuracy')
ax.set_title('rare-CpG suppression check: weighted vs. unweighted')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

fig_path = os.path.join(RESULTS_DIR, 'unweighted_check_comparison.png')
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f"Figure saved -> {fig_path}")
print("\n(results/cpg/comparison.png from notebook 04 is left unmodified -- this is a "
      "separate supplementary figure, since folding a 3rd (weighting) dimension into "
      "the existing 2D dataset x CpG grouped bar chart would clutter it rather than clarify it.)")


Figure saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\unweighted_check_comparison.png

(results/cpg/comparison.png from notebook 04 is left unmodified -- this is a separate supplementary figure, since folding a 3rd (weighting) dimension into the existing 2D dataset x CpG grouped bar chart would clutter it rather than clarify it.)
